In [1]:
import os
import re
import pandas as pd
import nltk
from pathlib import Path

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print(" All imports done!")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...


 All imports done!


[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
BASE_DIR   = Path(os.getcwd()).parent
INPUT_CSV  = BASE_DIR / "data" / "processed" / "emails_clean.csv"
OUTPUT_CSV = BASE_DIR / "data" / "processed" / "emails_preprocessed.csv"

print("Input :", INPUT_CSV)
print("Output:", OUTPUT_CSV)

Input : d:\deep_learning_projects\Smart_email_reply\smart_email_reply\data\processed\emails_clean.csv
Output: d:\deep_learning_projects\Smart_email_reply\smart_email_reply\data\processed\emails_preprocessed.csv


In [3]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df):,} emails")
df.head(3)

Loaded 8,064 emails


,sender,to,subject,date,body_clean
0,phillip.allen@enron.com,john.lavorato@enron.com,Re:,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",Traveling to have a business meeting takes the...
1,phillip.allen@enron.com,randall.gay@enron.com,NaN,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)","Randy,\n\n Can you send me a schedule of the s..."
2,phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,"Thu, 31 Aug 2000 04:17:00 -0700 (PDT)","Greg,\n\n How about either next Tuesday or Thu..."


In [5]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    
    # Step 1 - Lowercase
    text = text.lower()
    # Step 2 - Remove punctuation & numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Step 3 - Tokenize
    tokens = word_tokenize(text)
    # Step 4 - Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    # Step 5 - Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    return " ".join(tokens)

print("Preprocessing function ready!")

Preprocessing function ready!


In [6]:
print("Preprocessing emails")
df['body_processed'] = df['body_clean'].apply(preprocess_text)
print("Done!")
df[['body_clean', 'body_processed']].head(3)

Preprocessing emails
Done!


,body_clean,body_processed
0,Traveling to have a business meeting takes the...,traveling business meeting take fun trip espec...
1,"Randy,\n\n Can you send me a schedule of the s...",randy send schedule salary level everyone sche...
2,"Greg,\n\n How about either next Tuesday or Thu...",greg either next tuesday thursday phillip


In [7]:
print(f"Total emails     : {len(df):,}")
print(f"Empty rows       : {df['body_processed'].eq('').sum()}")

# Remove empty rows
df = df[df['body_processed'].str.len() > 0]
df.reset_index(drop=True, inplace=True)

print(f"After cleanup    : {len(df):,}")
print("\n sample:")
print("ORIGINAL :", df['body_clean'][0][:200])
print("\nPROCESSED:", df['body_processed'][0][:200])

Total emails     : 8,064
Empty rows       : 0
After cleanup    : 8,064

 sample:
ORIGINAL : Traveling to have a business meeting takes the fun out of the trip. Especially if you have to prepare a presentation. I would suggest holding the business plan meetings here then take a trip without a

PROCESSED: traveling business meeting take fun trip especially prepare presentation would suggest holding business plan meeting take trip without formal business meeting would even try get honest opinion whether


In [8]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved → {OUTPUT_CSV}")
print(f"Total rows: {len(df):,}")

Saved → d:\deep_learning_projects\Smart_email_reply\smart_email_reply\data\processed\emails_preprocessed.csv
Total rows: 8,064
